In [ ]:
# =============================================================================
# 情感分析（Sentiment Analysis）—— 数据预处理与加载
# =============================================================================
# 情感分析是自然语言处理中的二分类任务：判断一段文本的情感倾向（正面/负面）
# IMDb数据集：包含50,000条电影评论，25,000条训练 + 25,000条测试，正负样本各半
# 每条评论被标注为positive(1)或negative(0)

# 数据集
import os
import torch
from torch import nn
from d2l import torch as d2l

# =============================================================================
# 步骤1：配置数据集下载信息
# =============================================================================
# DATA_HUB是d2l库的数据集注册表，存储数据集名称到(URL, SHA1哈希)的映射
# 哈希值用于校验下载文件的完整性，防止下载损坏或被篡改的数据
#@save
d2l.DATA_HUB['aclImdb'] = (
    'http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz',
    '01ada507287d82875905620988597833ad4e0903')

# download_extract自动下载数据集并解压到指定子目录，利用缓存避免重复下载
# aclImdb是解压后的文件夹名称，包含train和test两个子目录
data_dir = d2l.download_extract('aclImdb', 'aclImdb')

# =============================================================================
# 步骤2：读取IMDb数据
# =============================================================================
# aclImdb目录结构：
# - train/pos/：训练集正面评论（文本文件）
# - train/neg/：训练集负面评论（文本文件）
# - test/pos/：测试集正面评论
# - test/neg/：测试集负面评论
#@save
def read_imdb(data_dir, is_train):
    """读取IMDb评论数据集文本序列和标签
    
    参数:
        data_dir: IMDb数据集根目录路径
        is_train: True读取训练集，False读取测试集
    返回:
        data: 评论文本列表，每个元素是一个字符串（评论内容）
        labels: 标签列表，1表示正面，0表示负面
    """
    data, labels = [], []
    # IMDb数据集按pos/neg两个文件夹组织，分别遍历
    for label in ('pos', 'neg'):
        # 根据is_train选择train或test目录
        folder_name = os.path.join(data_dir, 'train' if is_train else 'test',
                                   label)
        # 遍历该目录下的所有文件（每个文件是一条评论）
        for file in os.listdir(folder_name):
            # 以二进制模式读取，然后解码为UTF-8字符串
            # 'rb'模式读取原始字节，避免编码问题
            with open(os.path.join(folder_name, file), 'rb') as f:
                # 读取文件内容，解码为UTF-8，并移除换行符（\n会干扰文本处理）
                review = f.read().decode('utf-8').replace('\n', '')
                data.append(review)
                # pos文件夹的评论标签为1（正面），neg为0（负面）
                labels.append(1 if label == 'pos' else 0)
    return data, labels

# 读取训练集
train_data = read_imdb(data_dir, is_train=True)
print('训练集数目：', len(train_data[0]))
# 显示前3条样本及其标签，review[:60]只显示前60个字符
for x, y in zip(train_data[0][:3], train_data[1][:3]):
    print('标签：', y, 'review:', x[0:60])
    
# =============================================================================
# 步骤3：文本分词与构建词表
# =============================================================================
# tokenize将文本分割为词元（token）列表，token='word'表示按单词分词
# 例如："this movie is great" → ['this', 'movie', 'is', 'great']
train_tokens = d2l.tokenize(train_data[0], token='word')

# Vocab构建词表：将词元映射为整数索引
# min_freq=5：只保留出现频率≥5次的词，罕见词会增加词表大小且对模型帮助不大
# reserved_tokens=['<pad>']：保留特殊词元，<pad>用于序列填充（后面会用到）
vocab = d2l.Vocab(train_tokens, min_freq=5, reserved_tokens=['<pad>'])

# =============================================================================
# 步骤4：分析序列长度分布
# =============================================================================
d2l.set_figsize()  # 设置图表尺寸
d2l.plt.xlabel('# tokens per review')  # x轴：每条评论的词元数量
d2l.plt.ylabel('count')  # y轴：评论数量
# 绘制评论长度的直方图，bins=range(0, 1000, 50)表示0-50, 50-100, ...的区间
# 这帮助我们确定合适的序列长度截断值
d2l.plt.hist([len(line) for line in train_tokens], bins=range(0, 1000, 50));

# =============================================================================
# 步骤5：序列截断与填充（Truncation & Padding）
# =============================================================================
# RNN/CNN等模型需要固定长度的输入，因此需要统一序列长度
# num_steps=500：设置序列长度为500个词元
# 长度>500的评论会被截断（保留前500个词）
# 长度<500的评论会用<pad>填充到500
num_steps = 500  # 序列长度
train_features = torch.tensor([d2l.truncate_pad(
    vocab[line], num_steps, vocab['<pad>']) for line in train_tokens])
# truncate_pad(vocab[line], num_steps, vocab['<pad>'])：
#   - vocab[line]将词元列表转换为索引列表
#   - 如果长度>num_steps，截断到num_steps
#   - 如果长度<num_steps，用<pad>的索引填充到num_steps
print(train_features.shape)  # 输出应为(num_samples, 500)

# =============================================================================
# 步骤6：创建DataLoader
# =============================================================================
# load_array封装了TensorDataset和DataLoader的创建
# train_features是形状为(num_samples, 500)的特征矩阵
# train_data[1]是标签列表，转换为tensor
train_iter = d2l.load_array((train_features,
    torch.tensor(train_data[1])), 64)

# 检查一个batch的数据形状
for X, y in train_iter:
    print('X:', X.shape, ', y:', y.shape)  # X: (64, 500), y: (64,)
    break
print('小批量数目：', len(train_iter))

# =============================================================================
# 步骤7：封装完整的数据加载函数
# =============================================================================
#@save
def load_data_imdb(batch_size, num_steps=500):
    """返回数据迭代器和IMDb评论数据集的词表
    
    这是一个便捷函数，封装了数据下载、分词、词表构建、序列处理、DataLoader创建的全部流程
    
    参数:
        batch_size: 每个batch的样本数
        num_steps: 序列长度（默认500）
    返回:
        train_iter: 训练集DataLoader
        test_iter: 测试集DataLoader
        vocab: 词表对象
    """
    # 下载并解压数据集
    data_dir = d2l.download_extract('aclImdb', 'aclImdb')
    # 读取训练集和测试集
    train_data = read_imdb(data_dir, True)
    test_data = read_imdb(data_dir, False)
    # 分词
    train_tokens = d2l.tokenize(train_data[0], token='word')
    test_tokens = d2l.tokenize(test_data[0], token='word')
    # 构建词表（只用训练集构建，测试集出现的新词会被映射为<unk>未知词元）
    vocab = d2l.Vocab(train_tokens, min_freq=5)
    # 序列截断与填充
    train_features = torch.tensor([d2l.truncate_pad(
        vocab[line], num_steps, vocab['<pad>']) for line in train_tokens])
    test_features = torch.tensor([d2l.truncate_pad(
        vocab[line], num_steps, vocab['<pad>']) for line in test_tokens])
    # 创建DataLoader
    train_iter = d2l.load_array((train_features, torch.tensor(train_data[1])),
                                batch_size)
    test_iter = d2l.load_array((test_features, torch.tensor(test_data[1])),
                               batch_size,
                               is_train=False)  # 测试集不需要shuffle
    return train_iter, test_iter, vocab

In [ ]:
# =============================================================================
# 情感分析：使用双向LSTM（BiRNN）
# =============================================================================
# 本代码实现基于双向LSTM的情感分类模型
# 双向RNN的优势：同时捕获序列的前向和后向上下文信息
# 例如："not good"中的"not"修饰后面的"good"，双向结构能更好地理解这种依赖

import torch
from torch import nn
from d2l import torch as d2l

# =============================================================================
# 步骤1：加载数据
# =============================================================================
batch_size = 64
train_iter, test_iter, vocab = d2l.load_data_imdb(batch_size)

# =============================================================================
# 步骤2：定义双向LSTM模型
# =============================================================================
class BiRNN(nn.Module):
    """双向LSTM情感分类模型
    
    架构：Embedding → BiLSTM → Concat → Linear → Output
    
    关键设计：
    1. 使用双向LSTM同时捕获前向和后向上下文
    2. 连接序列首尾的隐状态作为文本表示
    3. 使用预训练GloVe词向量进行迁移学习
    """
    def __init__(self, vocab_size, embed_size, num_hiddens,
                 num_layers, **kwargs):
        """
        参数:
            vocab_size: 词表大小
            embed_size: 词嵌入维度
            num_hiddens: LSTM隐藏单元数（单向）
            num_layers: LSTM层数
        """
        super(BiRNN, self).__init__(**kwargs)
        # 词嵌入层：将词索引映射为密集向量
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 双向LSTM编码器：
        # - bidirectional=True启用双向模式
        # - 输出维度为2*num_hiddens（前向+后向拼接）
        self.encoder = nn.LSTM(embed_size, num_hiddens, num_layers=num_layers,
                                bidirectional=True)
        # 解码器：将编码后的表示映射为2类（正面/负面）
        # 输入维度是4*num_hiddens：连接初始和最终时间步的隐状态（各2*num_hiddens）
        self.decoder = nn.Linear(4 * num_hiddens, 2)

    def forward(self, inputs):
        """
        参数:
            inputs: 输入tensor，shape为(batch_size, num_steps)
        返回:
            outs: 输出tensor，shape为(batch_size, 2)
        """
        # inputs的形状是（批量大小，时间步数），例如(64, 500)
        # LSTM要求输入形状为(num_steps, batch_size, embed_size)，因此需要转置
        # embeddings的形状变为(num_steps, batch_size, embed_size)，例如(500, 64, 100)
        embeddings = self.embedding(inputs.T)
        
        # flatten_parameters()优化多GPU情况下的内存布局
        self.encoder.flatten_parameters()
        
        # outputs包含所有时间步的隐状态，shape为(num_steps, batch_size, 2*num_hiddens)
        # _包含最终的cell state和hidden state（本模型中不使用）
        outputs, _ = self.encoder(embeddings)
        
        # 关键设计：连接初始和最终时间步的隐状态
        # outputs[0]：第一个时间步的隐状态，捕获序列起始信息（后向LSTM的最终状态）
        # outputs[-1]：最后一个时间步的隐状态，捕获序列结束信息（前向LSTM的最终状态）
        # 两者拼接后形成完整的文本表示，shape为(batch_size, 4*num_hiddens)
        encoding = torch.cat((outputs[0], outputs[-1]), dim=1)
        
        # 通过全连接层分类，outs的shape为(batch_size, 2)
        outs = self.decoder(encoding)
        return outs
    
# =============================================================================
# 步骤3：模型配置与初始化
# =============================================================================
# 超参数设置
embed_size, num_hiddens, num_layers = 100, 100, 2
# 自动检测可用GPU设备
devices = d2l.try_all_gpus()
# 创建模型实例
net = BiRNN(len(vocab), embed_size, num_hiddens, num_layers)

# 权重初始化函数
def init_weights(m):
    """Xavier初始化：保持前向和反向传播时梯度的方差一致"""
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
    if type(m) == nn.LSTM:
        # LSTM有多个权重矩阵（W_ii, W_if, W_ig, W_io等），需要逐个初始化
        for param in m._flat_weights_names:
            if "weight" in param:
                nn.init.xavier_uniform_(m._parameters[param])

# 应用权重初始化
net.apply(init_weights);

# =============================================================================
# 步骤4：加载预训练GloVe词向量（迁移学习）
# =============================================================================
# GloVe（Global Vectors for Word Representation）是斯坦福NLP组发布的词向量
# glove.6b.100d表示使用60亿token语料训练的100维词向量
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')

# 根据词表获取对应的词向量
# embeds的shape为(vocab_size, embed_size)，即(num_tokens, 100)
embeds = glove_embedding[vocab.idx_to_token]
print(f"预训练词向量形状: {embeds.shape}")

# 将预训练词向量复制到模型的嵌入层
net.embedding.weight.data.copy_(embeds)
# 冻结嵌入层权重，使其在训练中不更新
# 理由：GloVe已经在大量语料上学习到了良好的词表示，不需要再微调
# 这也能防止过拟合，并加快训练速度
net.embedding.weight.requires_grad = False

# =============================================================================
# 步骤5：训练模型
# =============================================================================
lr, num_epochs = 0.01, 5
# Adam优化器：自适应学习率，结合了Momentum和RMSProp的优点
trainer = torch.optim.Adam(net.parameters(), lr=lr)
# 交叉熵损失函数，reduction="none"表示不自动求平均，便于自定义处理
loss = nn.CrossEntropyLoss(reduction="none")

# 使用d2l的训练函数进行训练
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
    devices)

# =============================================================================
# 步骤6：情感预测
# =============================================================================
# predict_sentiment函数在d2l库中定义，用于对输入文本进行情感预测
def predict_sentiment(net, vocab, sequence):
    """预测文本情感
    
    参数:
        net: 训练好的模型
        vocab: 词表
        sequence: 输入文本字符串
    返回:
        'positive'或'negative'
    """
    # 将文本分词并转换为词索引序列
    sequence = torch.tensor(vocab[sequence.split()], device=d2l.try_gpu())
    # 添加batch维度(1, seq_len)并送入模型
    label = torch.argmax(net(sequence.reshape(1, -1)), dim=1)
    return 'positive' if label == 1 else 'negative'

# 测试正面情感
print(f"'this movie is so great' -> {predict_sentiment(net, vocab, 'this movie is so great')}")
# 测试负面情感
print(f"'this movie is so bad' -> {predict_sentiment(net, vocab, 'this movie is so bad')}")

In [ ]:
# =============================================================================
# 情感分析：使用一维卷积神经网络（TextCNN）
# =============================================================================
# 本代码实现基于TextCNN的情感分类模型
# CNN的优势：通过不同大小的卷积核捕获不同粒度的n-gram特征
# 例如：3-gram卷积核能捕获"not very good"这样的短语模式

import torch
from torch import nn
from d2l import torch as d2l

# =============================================================================
# 步骤1：加载数据
# =============================================================================
batch_size = 64
train_iter, test_iter, vocab = d2l.load_data_imdb(batch_size)

# =============================================================================
# 步骤2：一维互相关运算（卷积的基础操作）
# =============================================================================
def corr1d(X, K):
    """一维互相关运算（单通道）
    
    参数:
        X: 输入序列，shape为(seq_len,)
        K: 卷积核，shape为(kernel_size,)
    返回:
        Y: 输出序列，shape为(seq_len - kernel_size + 1,)
    
    原理：卷积核在输入序列上滑动，逐位相乘再求和
    """
    w = K.shape[0]  # 卷积核大小
    # 输出长度 = 输入长度 - 卷积核大小 + 1
    Y = torch.zeros((X.shape[0] - w + 1))
    for i in range(Y.shape[0]):
        # 每个位置：将输入子序列与卷积核逐元素相乘，然后求和
        Y[i] = (X[i: i + w] * K).sum()
    return Y

# 示例：X = [0,1,2,3,4,5,6], K = [1,2]
# Y[0] = 0*1 + 1*2 = 2
# Y[1] = 1*1 + 2*2 = 5
# ...
X, K = torch.tensor([0, 1, 2, 3, 4, 5, 6]), torch.tensor([1, 2])
print("一维卷积示例:", corr1d(X, K))

# =============================================================================
# 步骤3：多输入通道的一维互相关运算
# =============================================================================
def corr1d_multi_in(X, K):
    """多输入通道的一维互相关运算
    
    参数:
        X: 输入张量，shape为(num_channels, seq_len)
        K: 卷积核张量，shape为(num_channels, kernel_size)
    返回:
        输出序列，shape为(seq_len - kernel_size + 1,)
    
    实现：对每个通道分别做卷积，然后将结果相加（多通道信息融合）
    """
    # zip(X, K)将输入和卷积核按通道配对
    # 对每个通道调用corr1d，然后用sum将所有通道的结果相加
    return sum(corr1d(x, k) for x, k in zip(X, K))

# 示例：3个输入通道，每个通道长度为7，卷积核大小为2
X = torch.tensor([[0, 1, 2, 3, 4, 5, 6],
              [1, 2, 3, 4, 5, 6, 7],
              [2, 3, 4, 5, 6, 7, 8]])
K = torch.tensor([[1, 2], [3, 4], [-1, -3]])
print("多通道一维卷积示例:", corr1d_multi_in(X, K))

# =============================================================================
# 步骤4：定义TextCNN模型
# =============================================================================
class TextCNN(nn.Module):
    """TextCNN情感分类模型
    
    架构：
    输入 → [可训练Embedding + 固定Embedding] → Conv1d(多种核) → MaxPool → Concat → FC → Output
    
    关键设计：
    1. 双通道嵌入：一个可训练，一个固定（预训练），捕获不同层面的语义信息
    2. 多尺度卷积核：同时使用3、4、5-gram等卷积核，捕获不同长度的短语特征
    3. 最大池化：对每个卷积结果取max，提取最重要的激活特征
    """
    def __init__(self, vocab_size, embed_size, kernel_sizes, num_channels,
                 **kwargs):
        """
        参数:
            vocab_size: 词表大小
            embed_size: 词嵌入维度
            kernel_sizes: 卷积核大小列表，如[3, 4, 5]
            num_channels: 每个卷积核对应的输出通道数列表，如[100, 100, 100]
        """
        super(TextCNN, self).__init__(**kwargs)
        # 可训练的词嵌入层：在训练过程中会更新
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 固定的词嵌入层：使用预训练词向量，不更新（constant）
        # 双通道设计的好处：可训练层可以学习任务特定的表示，固定层保留通用的语义信息
        self.constant_embedding = nn.Embedding(vocab_size, embed_size)
        # Dropout防止过拟合，在训练时以0.5的概率随机丢弃神经元
        self.dropout = nn.Dropout(0.5)
        # 解码器：将拼接后的特征映射为2类
        # 输入维度是sum(num_channels)，因为每个卷积核产生num_channels个输出通道
        self.decoder = nn.Linear(sum(num_channels), 2)
        # 自适应平均池化：将每个通道的序列长度压缩为1（即全局平均池化）
        # 这里实际上使用的是max pooling，但名字是AvgPool（d2l书中的实现）
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.relu = nn.ReLU()
        
        # 创建多个一维卷积层，每个对应一个kernel_size
        self.convs = nn.ModuleList()
        for c, k in zip(num_channels, kernel_sizes):
            # 输入通道数 = 2 * embed_size（两个嵌入层拼接后的维度）
            # 输出通道数 = c（该卷积核对应的通道数）
            # 卷积核大小 = k（在序列维度上滑动，捕获k-gram特征）
            self.convs.append(nn.Conv1d(2 * embed_size, c, k))

    def forward(self, inputs):
        """
        参数:
            inputs: 输入tensor，shape为(batch_size, num_steps)
        返回:
            outputs: 输出tensor，shape为(batch_size, 2)
        """
        # 1. 双通道嵌入并拼接
        # 两个嵌入层的输出都是(batch_size, num_steps, embed_size)
        # 在dim=2（嵌入维度）上拼接后，shape为(batch_size, num_steps, 2*embed_size)
        embeddings = torch.cat((
            self.embedding(inputs), self.constant_embedding(inputs)), dim=2)
        
        # 2. 调整维度以适应Conv1d的输入格式
        # Conv1d期望输入为(batch_size, channels, seq_len)
        # 从(batch, seq_len, 2*embed) permute为(batch, 2*embed, seq_len)
        embeddings = embeddings.permute(0, 2, 1)
        
        # 3. 多尺度卷积 + 池化
        # 对每个卷积层：
        #   - conv(embeddings)输出shape为(batch, num_channels, seq_len-k+1)
        #   - ReLU激活引入非线性
        #   - pool将seq_len压缩为1，输出shape为(batch, num_channels, 1)
        #   - squeeze移除最后一个维度，shape为(batch, num_channels)
        # 最后将所有卷积结果在dim=1上拼接
        encoding = torch.cat([
            torch.squeeze(self.relu(self.pool(conv(embeddings))), dim=-1)
            for conv in self.convs], dim=1)
        
        # 4. Dropout + 全连接分类
        outputs = self.decoder(self.dropout(encoding))
        return outputs
    
# =============================================================================
# 步骤5：模型配置与初始化
# =============================================================================
# 超参数：使用3种卷积核（3-gram, 4-gram, 5-gram），每种100个输出通道
# 这些超参数是TextCNN论文中的经典配置
embed_size, kernel_sizes, nums_channels = 100, [3, 4, 5], [100, 100, 100]
devices = d2l.try_all_gpus()
net = TextCNN(len(vocab), embed_size, kernel_sizes, nums_channels)

# 权重初始化
def init_weights(m):
    """Xavier初始化"""
    if type(m) in (nn.Linear, nn.Conv1d):
        nn.init.xavier_uniform_(m.weight)

net.apply(init_weights);

# =============================================================================
# 步骤6：加载预训练GloVe词向量
# =============================================================================
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')
embeds = glove_embedding[vocab.idx_to_token]
# 将预训练词向量复制到两个嵌入层
net.embedding.weight.data.copy_(embeds)
net.constant_embedding.weight.data.copy_(embeds)
# 固定constant_embedding层
net.constant_embedding.weight.requires_grad = False

# =============================================================================
# 步骤7：训练模型
# =============================================================================
lr, num_epochs = 0.001, 5  # CNN使用较小的学习率
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction="none")
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs, devices)

# =============================================================================
# 步骤8：情感预测
# =============================================================================
# 使用d2l库提供的predict_sentiment函数
print("\n测试模型情感预测能力:")
print(f"'this movie is so great' -> {d2l.predict_sentiment(net, vocab, 'this movie is so great')}")
print(f"'this movie is so bad' -> {d2l.predict_sentiment(net, vocab, 'this movie is so bad')}")

In [ ]:
# =============================================================================
# 情感分析：循环神经网络（增强版，带详细训练监控）
# =============================================================================
# 本代码是BiRNN的增强版本，添加了详细的训练日志、进度条、模型检查点等功能
# 适用于需要深入观察训练过程和调试的场景

import torch
from torch import nn
from d2l import torch as d2l
import time
from tqdm import tqdm

# =============================================================================
# 步骤1：训练日志记录
# =============================================================================
print("="*50)
print(f"开始情感分析训练 - {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*50)

# =============================================================================
# 步骤2：加载数据
# =============================================================================
batch_size = 64
print(f"[数据加载] 开始加载IMDB数据集，批大小: {batch_size}")
train_iter, test_iter, vocab = d2l.load_data_imdb(batch_size)
print(f"[数据加载] 完成! 词汇表大小: {len(vocab)}")

# =============================================================================
# 步骤3：定义双向LSTM模型（与基础版相同）
# =============================================================================
class BiRNN(nn.Module):
    """双向LSTM情感分类模型"""
    def __init__(self, vocab_size, embed_size, num_hiddens,
                 num_layers, **kwargs):
        super(BiRNN, self).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 将bidirectional设置为True以获取双向循环神经网络
        self.encoder = nn.LSTM(embed_size, num_hiddens, num_layers=num_layers,
                                bidirectional=True)
        self.decoder = nn.Linear(4 * num_hiddens, 2)

    def forward(self, inputs):
        # inputs的形状是（批量大小，时间步数）
        # 因为长短期记忆网络要求其输入的第一个维度是时间维，
        # 所以在获得词元表示之前，输入会被转置。
        # 输出形状为（时间步数，批量大小，词向量维度）
        embeddings = self.embedding(inputs.T)
        self.encoder.flatten_parameters()
        # 返回上一个隐藏层在不同时间步的隐状态，
        # outputs的形状是（时间步数，批量大小，2*隐藏单元数）
        outputs, _ = self.encoder(embeddings)
        # 连结初始和最终时间步的隐状态，作为全连接层的输入，
        # 其形状为（批量大小，4*隐藏单元数）
        encoding = torch.cat((outputs[0], outputs[-1]), dim=1)
        outs = self.decoder(encoding)
        return outs
    
# =============================================================================
# 步骤4：模型配置与创建
# =============================================================================
embed_size, num_hiddens, num_layers = 100, 100, 2
print(f"[模型配置] 嵌入大小: {embed_size}, 隐藏单元数: {num_hiddens}, LSTM层数: {num_layers}")

print("[设备检测] 正在检测可用设备...")
devices = d2l.try_all_gpus()
print(f"[设备检测] 可用设备: {devices}")

net = BiRNN(len(vocab), embed_size, num_hiddens, num_layers)
# 计算模型参数量：sum(p.numel() for p in net.parameters())
# numel()返回tensor中元素总数，即该参数的参数量
print(f"[模型创建] 双向RNN模型已创建，参数量: {sum(p.numel() for p in net.parameters()):,}")

# =============================================================================
# 步骤5：权重初始化
# =============================================================================
def init_weights(m):
    """Xavier均匀初始化"""
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
    if type(m) == nn.LSTM:
        for param in m._flat_weights_names:
            if "weight" in param:
                nn.init.xavier_uniform_(m._parameters[param])

print("[初始化] 正在初始化模型权重...")
net.apply(init_weights)
print("[初始化] 完成!")

# =============================================================================
# 步骤6：加载预训练词向量
# =============================================================================
print("[词嵌入] 正在加载GloVe词向量...")
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')
embeds = glove_embedding[vocab.idx_to_token]
print(f"[词嵌入] 加载完成! 形状: {embeds.shape}")

print("[词嵌入] 将预训练词向量复制到嵌入层...")
net.embedding.weight.data.copy_(embeds)
net.embedding.weight.requires_grad = False
print("[词嵌入] 完成! 嵌入层权重已冻结")

# =============================================================================
# 步骤7：训练配置
# =============================================================================
lr, num_epochs = 0.01, 5
print(f"[训练配置] 学习率: {lr}, 训练轮数: {num_epochs}")

trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction="none")
print(f"[优化器] 使用Adam优化器，学习率: {lr}")
print(f"[损失函数] 使用交叉熵损失函数")

# =============================================================================
# 步骤8：增强版训练函数
# =============================================================================
def train_ch13_with_progress(net, train_iter, test_iter, loss, trainer, num_epochs, devices):
    """训练模型并显示详细进度
    
    相比d2l.train_ch13，本函数添加了：
    - tqdm进度条显示训练进度
    - 每轮训练和测试的详细耗时统计
    - 模型检查点保存（当测试准确率>0.85时）
    - 最佳模型追踪
    """
    print("\n" + "="*50)
    print(f"开始训练模型 (共 {num_epochs} 轮)")
    print("="*50)
    
    # 将模型移动到设备（通常是GPU）
    net.to(devices[0])
    
    # 记录最佳准确率和对应的轮次
    best_test_acc = 0.0
    best_epoch = 0
    
    # 训练循环
    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f"\n[Epoch {epoch+1}/{num_epochs}] 开始训练...")
        
        # 设置模型为训练模式（启用Dropout等）
        net.train()
        # Accumulator(3)用于累加：训练损失总和、训练准确率总和、样本数
        metric = d2l.Accumulator(3)
        train_batches = len(train_iter)
        
        # 使用tqdm创建进度条，实时显示训练进度
        with tqdm(total=train_batches, desc=f"训练轮次 {epoch+1}/{num_epochs}") as pbar:
            for i, (X, y) in enumerate(train_iter):
                # 梯度清零：防止梯度累积
                trainer.zero_grad()
                # 将数据移动到GPU
                X, y = X.to(devices[0]), y.to(devices[0])
                # 前向传播
                y_hat = net(X)
                # 计算损失（reduction="none"返回每个样本的损失）
                l = loss(y_hat, y)
                # 反向传播：计算梯度
                l.mean().backward()
                # 参数更新
                trainer.step()
                
                # 更新指标（使用no_grad避免构建计算图，节省内存）
                with torch.no_grad():
                    metric.add(float(l.sum()), d2l.accuracy(y_hat, y), y.numel())
                
                # 每10个批次更新一次进度条显示的信息
                if i % 10 == 0 or i == train_batches - 1:
                    train_loss = metric[0] / metric[2]
                    train_acc = metric[1] / metric[2]
                    pbar.set_postfix({
                        'loss': f'{train_loss:.4f}',
                        'acc': f'{train_acc:.3f}',
                        'samples': f'{metric[2]:,}'
                    })
                
                pbar.update(1)
        
        # 计算训练指标
        train_loss = metric[0] / metric[2]
        train_acc = metric[1] / metric[2]
        
        # 测试阶段
        test_start_time = time.time()
        print(f"[Epoch {epoch+1}/{num_epochs}] 训练完成! 开始测试...")
        # evaluate_accuracy_gpu计算测试集准确率
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter)
        
        # 更新最佳准确率记录
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch + 1
        
        # 计算耗时
        epoch_time = time.time() - epoch_start_time
        test_time = time.time() - test_start_time
        
        # 打印本轮结果
        print(f"[Epoch {epoch+1}/{num_epochs}] 结果:")
        print(f"  训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.3f}")
        print(f"  测试准确率: {test_acc:.3f} | 测试时间: {test_time:.1f}s")
        print(f"  本轮总耗时: {epoch_time:.1f}s")
        
        # 保存模型检查点（只保存表现较好的模型以节省磁盘空间）
        if test_acc > 0.85:
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': net.state_dict(),  # 模型参数
                'optimizer_state_dict': trainer.state_dict(),  # 优化器状态（可用于恢复训练）
                'loss': train_loss,
                'accuracy': test_acc
            }
            torch.save(checkpoint, f'model_epoch_{epoch+1}_acc_{test_acc:.3f}.pth')
            print(f"  模型已保存为 'model_epoch_{epoch+1}_acc_{test_acc:.3f}.pth'")
    
    # 训练结束总结
    print("\n" + "="*50)
    print(f"训练完成! 总耗时: {time.time() - start_time:.1f}秒")
    print(f"最佳测试准确率: {best_test_acc:.3f} (在第 {best_epoch} 轮)")
    print("="*50)
    return train_loss, train_acc, test_acc

# =============================================================================
# 步骤9：执行训练
# =============================================================================
# 记录开始时间
start_time = time.time()

# 使用增强版训练函数
train_ch13_with_progress(net, train_iter, test_iter, loss, trainer, num_epochs, devices)

# =============================================================================
# 步骤10：情感预测测试
# =============================================================================
def predict_sentiment(net, vocab, sequence):
    """预测文本情感
    
    参数:
        net: 训练好的模型
        vocab: 词表
        sequence: 输入文本字符串
    返回:
        sentiment: 'positive'或'negative'
    """
    # 分词并转换为词索引序列
    sequence = torch.tensor(vocab[sequence.split()], device=d2l.try_gpu())
    # 添加batch维度并送入模型，取概率最大的类别
    label = torch.argmax(net(sequence.reshape(1, -1)), dim=1)
    sentiment = 'positive' if label == 1 else 'negative'
    print(f"文本: '{sequence}'\n预测情感: {sentiment} (标签: {label.item()})")
    return sentiment

# 测试模型
print("\n测试模型情感预测能力:")
print("="*30)
predict_sentiment(net, vocab, 'this movie is so great')
print("-"*30)
predict_sentiment(net, vocab, 'this movie is so bad')
print("="*30)